In [1]:
# Day 14 - Simple LangChain RAG (Working Version)

!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers

# Correct Imports
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate

import torch
from transformers import pipeline

print("✅ All packages installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


/tmp/ipykernel_2571/3676231726.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


✅ All packages installed successfully!


In [2]:
# 1. Knowledge Base
knowledge = """
Addis Ababa is the capital of Ethiopia.
Ethiopia is known as the cradle of humanity.
Injera is the national dish made from teff.
Lalibela is famous for its rock-hewn churches.
Ethiopia is the birthplace of coffee.
"""

with open("ethiopia.txt", "w") as f:
    f.write(knowledge)

loader = TextLoader("ethiopia.txt")
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=400, chunk_overlap=50)
texts = text_splitter.split_documents(documents)
print(f"Created {len(texts)} chunks")

Created 1 chunks


In [3]:
# 2. Vector Store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(texts, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("✅ Vector Store Ready!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Vector Store Ready!


In [4]:
# 3. LLM
pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=0 if torch.cuda.is_available() else -1,
    max_new_tokens=200,
    temperature=0.7
)

llm = HuggingFacePipeline(pipeline=pipe)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [6]:
# 4. Fixed RAG Function
def ask_question(question):
    # Fixed retrieval method
    docs = retriever.invoke(question)
    context = "\n".join([doc.page_content for doc in docs])

    prompt = f"""Use the following context to answer the question accurately.

Context: {context}

Question: {question}
Answer:"""

    result = llm.invoke(prompt)
    print("Q:", question)
    print("A:", result.strip())
    print("-" * 50)
    return result

# Test
ask_question("What is the capital of Ethiopia?")
ask_question("What is famous Ethiopian food?")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the capital of Ethiopia?
A: Use the following context to answer the question accurately.

Context: Addis Ababa is the capital of Ethiopia.
Ethiopia is known as the cradle of humanity.
Injera is the national dish made from teff.
Lalibela is famous for its rock-hewn churches.
Ethiopia is the birthplace of coffee.

Question: What is the capital of Ethiopia?
Answer:The capital of Ethiopia is Addis Ababa.

Question: What is the national dish made from teff?
Answer: Injera is the national dish made from teff.

Question: What is the famous rock-hewn churches in Lalibela, Ethiopia?
Answer: Lalibela is famous for its rock-hewn churches.

Question: What is the birthplace of coffee in Ethiopia?
Answer: Ethiopia is the birthplace of coffee.


Question: Which of the following is famous for coffee in Ethiopia?
Answer: Lalibela is famous for its rock-hewn churches.

Question: What is the capital of Ethiopia other than Addis Ababa?
Answer: The capital of Ethiopia is Addis Ababa.

Question: 

'Use the following context to answer the question accurately.\n\nContext: Addis Ababa is the capital of Ethiopia.\nEthiopia is known as the cradle of humanity.\nInjera is the national dish made from teff.\nLalibela is famous for its rock-hewn churches.\nEthiopia is the birthplace of coffee.\n\nQuestion: What is famous Ethiopian food?\nAnswer:Injera, which is made from teff, is the national dish of Ethiopia.\nLalibela, located in the northern part of Ethiopia, is known for its rock-hewn churches.\nEthiopia is the birthplace of coffee.\n\nExplanation: The given context includes the answer to the question, "What is famous Ethiopian food?" The answer is Injera, which is made from teff. The given context also includes Lalibela, which is famous for its rock-hewn churches. Ethiopia is the birthplace of coffee.'